Initialize Models

In [ ]:
from transformers import AutoTokenizer
from sentence_transformers import CrossEncoder

model = CrossEncoder("similarity-minilm/checkpoint-20216", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

Prepare training set

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset('csv', data_files='./Language Challenge/quora-question-pairs/train.csv')['train']

def preprocess_train(example):
  sentence_1 = [str(x) for x in example['question1']]
  sentence_2 = [str(x) for x in example['question2']]

  return {"sentence1": sentence_1, "sentence2": sentence_2, "label": example['is_duplicate']}

train_dataset = train_dataset.map(preprocess_train, batched=True, remove_columns = train_dataset.column_names)
train_dataset

Training of the model

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoderTrainingArguments, CrossEncoderTrainer

training_args = CrossEncoderTrainingArguments(
    output_dir="./similarity-minilm",
    learning_rate=5e-5,
    num_train_epochs=4,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=1,
    fp16=True
)

trainer = CrossEncoderTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

Calculation of F1-Score

In [ ]:
true_pos, false_pos, false_neg = 0,0,0
for i, element in enumerate(model.predict(list(zip(train_dataset['sentence1'], train_dataset['sentence2'])))):
    logit = 0 if element[0] > element[1] else 1

    if logit==1:
        if logit==train_dataset[i]['label']:
            true_pos+=1;
        else:
            false_pos+=1;
    elif logit!=train_dataset[i]['label']:
        false_neg+=1    

precision = true_pos/(true_pos + false_pos)
recall = true_pos/(true_pos + false_neg)
    

In [ ]:
print(f"F1 Score: {2*precision*recall/(precision + recall)}")